In [ ]:
# Import Required Libraries

import warnings
warnings.filterwarnings("ignore")

import os
import joblib

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.model_selection import (
    RandomizedSearchCV,
    cross_val_score
)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.3f}".format)

print("Libraries Imported Successfully")

In [ ]:
# Load Processed Datasets

X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")

X_train_scaled = pd.read_csv(
    "../data/processed/X_train_scaled.csv"
)

X_test_scaled = pd.read_csv(
    "../data/processed/X_test_scaled.csv"
)

y_train = pd.read_csv("../data/processed/y_train.csv")

y_test = pd.read_csv("../data/processed/y_test.csv")

print("Processed Datasets Loaded Successfully")

In [ ]:
# Dataset Verification

print("Training Feature Matrix :", X_train.shape)
print("Testing Feature Matrix  :", X_test.shape)

print()

print("Training Target :", y_train.shape)
print("Testing Target  :", y_test.shape)

In [ ]:
# Remove Identifier Columns

identifier_columns = [
    "Run",
    "Event"
]

X_train = X_train.drop(
    columns=identifier_columns
)

X_test = X_test.drop(
    columns=identifier_columns
)

X_train_scaled = X_train_scaled.drop(
    columns=identifier_columns
)

X_test_scaled = X_test_scaled.drop(
    columns=identifier_columns
)

print("Identifier Columns Removed Successfully")

In [ ]:
# Convert Target Variables to Series

y_train = y_train.squeeze()

y_test = y_test.squeeze()

print("Target Variables Converted Successfully")

In [ ]:
# Import Regression Models

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)

from sklearn.tree import (
    DecisionTreeRegressor
)

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor
)

from xgboost import (
    XGBRegressor
)

from lightgbm import (
    LGBMRegressor
)

from catboost import (
    CatBoostRegressor
)

print("Regression Models Imported Successfully")

In [ ]:
# Initialize Regression Models

models = {

    "Linear Regression":
        LinearRegression(),

    "Ridge":
        Ridge(),

    "Lasso":
        Lasso(),

    "ElasticNet":
        ElasticNet(),

    "Decision Tree":
        DecisionTreeRegressor(
            random_state=42
        ),

    "Random Forest":
        RandomForestRegressor(
            random_state=42,
            n_jobs=-1
        ),

    "Extra Trees":
        ExtraTreesRegressor(
            random_state=42,
            n_jobs=-1
        ),

    "Gradient Boosting":
        GradientBoostingRegressor(
            random_state=42
        ),

    "XGBoost":
        XGBRegressor(
            random_state=42,
            n_jobs=-1,
            verbosity=0
        ),

    "LightGBM":
        LGBMRegressor(
            random_state=42,
            verbosity=-1
        ),

    "CatBoost":
        CatBoostRegressor(
            random_state=42,
            verbose=0
        )

}

print("Regression Models Initialized Successfully")

In [ ]:
# Evaluate Regression Model

def evaluate_model(

    model,
    X_train_data,
    X_test_data,
    y_train,
    y_test

):

    # Train the model
    model.fit(
        X_train_data,
        y_train
    )

    # Predict on the test set
    predictions = model.predict(
        X_test_data
    )

    # Evaluation Metrics
    mae = mean_absolute_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            predictions
        )
    )

    r2 = r2_score(
        y_test,
        predictions
    )

    n = X_test_data.shape[0]
    p = X_test_data.shape[1]

    adjusted_r2 = 1 - (
        (1 - r2) *
        (n - 1) /
        (n - p - 1)
    )

    return (
        mae,
        rmse,
        r2,
        adjusted_r2,
        predictions
    )

In [ ]:
# Train Baseline Models

results = []

trained_models = {}

predictions_dict = {}

scaled_models = [

    "Linear Regression",
    "Ridge",
    "Lasso",
    "ElasticNet"

]

for model_name, model in models.items():

    # Select the appropriate dataset

    if model_name in scaled_models:

        X_train_data = X_train_scaled
        X_test_data = X_test_scaled

    else:

        X_train_data = X_train
        X_test_data = X_test

    # Evaluate model

    mae, rmse, r2, adjusted_r2, predictions = evaluate_model(

        model,
        X_train_data,
        X_test_data,
        y_train,
        y_test

    )

    # Store trained model

    trained_models[model_name] = model

    # Store predictions

    predictions_dict[model_name] = {

        "Actual": y_test,

        "Predicted": predictions,

        "X_test": X_test_data

    }

    # Store evaluation metrics

    results.append({

        "Model": model_name,

        "MAE": mae,

        "RMSE": rmse,

        "R²": r2,

        "Adjusted R²": adjusted_r2

    })

print("Baseline Models Trained Successfully")

In [ ]:
# Create Model Comparison Table

results_df = pd.DataFrame(
    results
)

results_df = results_df.sort_values(

    by="RMSE",

    ascending=True

).reset_index(
    drop=True
)

display(results_df)

In [ ]:
# Best Baseline Model

best_model_name = results_df.iloc[0]["Model"]

print("Best Baseline Model :", best_model_name)

print()

display(
    results_df.head()
)

In [ ]:
# Model Performance Comparison

display(results_df)

In [ ]:
# Compare Models Using RMSE

plt.figure(figsize=(10, 6))

sns.barplot(

    data=results_df,

    x="RMSE",

    y="Model"

)

plt.title("RMSE Comparison of Regression Models")

plt.xlabel("RMSE")

plt.ylabel("Model")

plt.tight_layout()

plt.show()

In [ ]:
# Compare Models Using MAE

plt.figure(figsize=(10, 6))

sns.barplot(

    data=results_df,

    x="MAE",

    y="Model"

)

plt.title("MAE Comparison of Regression Models")

plt.xlabel("MAE")

plt.ylabel("Model")

plt.tight_layout()

plt.show()

In [ ]:
# Compare Models Using R²

plt.figure(figsize=(10, 6))

sns.barplot(

    data=results_df,

    x="R²",

    y="Model"

)

plt.title("R² Comparison of Regression Models")

plt.xlabel("R²")

plt.ylabel("Model")

plt.tight_layout()

plt.show()

In [ ]:
# Compare Models Using Adjusted R²

plt.figure(figsize=(10, 6))

sns.barplot(

    data=results_df,

    x="Adjusted R²",

    y="Model"

)

plt.title("Adjusted R² Comparison of Regression Models")

plt.xlabel("Adjusted R²")

plt.ylabel("Model")

plt.tight_layout()

plt.show()

In [ ]:
# Top Performing Models

display(

    results_df.head()

)

In [ ]:
# Save Baseline Model Results

os.makedirs(

    "../results",

    exist_ok=True

)

results_df.to_csv(

    "../results/baseline_model_results.csv",

    index=False

)

print("Baseline Results Saved Successfully")